# Variability and Convergence of Gherkin Generations with Normalized Token-Level Levenshtein Distance

This notebook receives **one generation JSON file** in the following format:

- `model`
- `technique`
- `number_of_executions`
- `cases[]`
  - `case_id`
  - `original_case`
  - `generations[]`
    - `execution`
    - `gherkin`

## Objective

The analysis measures **textual variation across different executions of the same base case**.

There is no comparison between different cases and no comparison with a reference. For each `case_id`, the different generations are compared **pairwise**.

## Metric

Each Gherkin text is converted into a sequence of tokens:

\[
T(G) = [t_1,t_2,\ldots,t_k]
\]

Token-level Levenshtein distance corresponds to the minimum number of:

- token insertions;
- token deletions;
- token substitutions;

required to transform one sequence into the other.

The normalized distance used in this notebook is:

\[
D_{NTL}(A,B)=
\frac{Lev(T(A),T(B))}
{\max(|T(A)|,|T(B)|)}
\]

with:

\[
0 \le D_{NTL}(A,B) \le 1
\]

Interpretation:

- `0` → identical token sequences after the configured textual normalization;
- values close to `0` → low textual variation;
- higher values → greater textual variation;
- `1` → maximum edit distance for that pair under this normalization.

> The metric measures **textual variation**, not semantic equivalence. Different synonyms remain different tokens.

## Variability per Case

For a case \(c\), considering the first \(n\) executions:

\[
V_c(n)=
\frac{1}{\binom{n}{2}}
\sum_{i<j}
D_{NTL}(G_{c,i},G_{c,j})
\]

## Global Variability of the Configuration

If there are \(C\) base cases:

\[
V(n)=
\frac{1}{C}
\sum_{c=1}^{C}V_c(n)
\]

Thus, **each base case receives the same weight** in the global estimate.

## Convergence

Because the metric is on a fixed \([0,1]\) scale, the notebook uses **absolute tolerance**.

If \(N\) is the total number of available executions:

\[
V_{final}=V(N)
\]

The convergence point is the smallest \(n\) for which:

\[
|V(m)-V(N)| \le \epsilon
\quad \forall m \ge n
\]

In addition, the notebook requires a minimum number of subsequent executions after the candidate point, preventing the final execution itself from being considered a trivial convergence point.

By default:

- `MIN_CONVERGENCE_N = 10`;
- `ABSOLUTE_TOLERANCE = 0.01`;
- `MIN_SUBSEQUENT_EXECUTIONS = 5`.

Therefore, with 20 executions, a point at `n = 15` can still be confirmed because there are 5 subsequent executions (`16...20`). A point at `n = 18`, for example, will be considered **unconfirmed with N=20**, because there are not enough subsequent observations.


In [ ]:

# Dependency installation
# RapidFuzz implements Levenshtein efficiently.

%pip install -q rapidfuzz pandas numpy matplotlib


In [ ]:

# =========================
# ANALYSIS CONFIGURATION
# =========================

# Colab: leave as None to open the upload selector.
# Jupyter/local: provide the JSON file path.
JSON_FILE = None

# -------------------------
# Normalization/tokenization
# -------------------------

# Ignore differences only in uppercase/lowercase.
CONVERT_TO_LOWERCASE = True

# Keep punctuation as separate tokens.
# E.g.: ":" and quotation marks may also contribute to textual distance.
INCLUDE_PUNCTUATION = True

# -------------------------
# Convergence criterion
# -------------------------

MIN_CONVERGENCE_N = 10

# Because the metric ranges from 0 to 1, we use ABSOLUTE difference.
# E.g.: 0.01 means that V(n) must remain at most
# 0.01 units away from the estimate obtained with all executions.
#
# This is an operational precision criterion for the study,
# not a universal threshold.
ABSOLUTE_TOLERANCE = 0.01

# Prevents declaring convergence merely because the final point was reached.
# The candidate must have at least this number of subsequent executions.
MIN_SUBSEQUENT_EXECUTIONS = 5

# -------------------------
# Bootstrap CI over CASES
# -------------------------

CALCULATE_BOOTSTRAP = True
N_BOOTSTRAP = 1000
SEED_BOOTSTRAP = 2026

# Number of cases displayed in rankings/plots.
TOP_CASES = 20

# Optional sensitivity analysis for the tolerance.
SENSITIVITY_TOLERANCES = [0.005, 0.01, 0.02]


In [ ]:

import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rapidfuzz.distance import Levenshtein

pd.set_option("display.max_colwidth", 120)


In [ ]:

# =========================
# JSON INPUT
# =========================

if JSON_FILE is None:
    try:
        from google.colab import files
        uploaded = files.upload()

        if not uploaded:
            raise ValueError("No file was uploaded.")

        JSON_FILE = next(iter(uploaded.keys()))

    except ImportError:
        raise ValueError(
            "Outside Google Colab, set JSON_FILE to the JSON file path."
        )

with open(JSON_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

required_fields = {
    "model",
    "technique",
    "number_of_executions",
    "cases"
}

missing_fields = required_fields - set(data.keys())

if missing_fields:
    raise ValueError(
        f"Required fields missing from the JSON: {sorted(missing_fields)}"
    )

llm_model = data["model"]
technique = data["technique"]
declared_executions = int(data["number_of_executions"])
cases = data["cases"]

print("File:", JSON_FILE)
print("LLM model:", llm_model)
print("Technique:", technique)
print("Cases:", len(cases))
print("Declared executions:", declared_executions)


In [ ]:

# =========================
# STRUCTURE VALIDATION
# =========================

records = []
problems = []

for case in cases:

    case_id = case.get("case_id")
    original_case = case.get("original_case", "")
    generations = case.get("generations", [])

    executions = sorted(
        [
            g for g in generations
            if isinstance(g.get("gherkin"), str)
            and g.get("gherkin").strip()
        ],
        key=lambda x: int(x["execution"])
    )

    execution_ids = [
        int(g["execution"])
        for g in executions
    ]

    if len(executions) != declared_executions:
        problems.append(
            f"{case_id}: {len(executions)} valid generations; "
            f"expected {declared_executions}."
        )

    if execution_ids != list(
        range(1, len(executions) + 1)
    ):
        problems.append(
            f"{case_id}: execution numbering is not sequential: "
            f"{execution_ids[:10]}..."
        )

    for g in executions:

        records.append({
            "case_id": case_id,
            "original_case": original_case,
            "execution": int(g["execution"]),
            "generation_id": g.get("generation_id"),
            "gherkin": g["gherkin"].strip()
        })

if problems:

    print("Problems found:")

    for p in problems[:20]:
        print("-", p)

    raise ValueError(
        "The file contains incomplete cases or inconsistent numbering. "
        "Correct the JSON before the analysis."
    )

df = pd.DataFrame(records)

counts = (
    df.groupby("case_id")["execution"]
    .nunique()
)

if counts.nunique() != 1:
    raise ValueError(
        "The cases do not have the same number of executions."
    )

MAX_N = int(counts.iloc[0])

if MAX_N != declared_executions:
    raise ValueError(
        f"The JSON declares {declared_executions} executions, "
        f"but {MAX_N} were found per case."
    )

print(
    f"Valid structure: "
    f"{df['case_id'].nunique()} cases × {MAX_N} executions."
)

display(df.head())


## Tokenization Used

The analysis does not use embeddings.

Before calculating Levenshtein distance, the notebook applies only minimal normalization:

1. Unicode `NFKC` normalization;
2. optional conversion to lowercase;
3. tokenization;
4. spaces and line breaks are not counted as tokens;
5. punctuation may be retained as a token, according to the configuration.

The following are not applied:

- stemming;
- lemmatization;
- stopword removal;
- synonym substitution;
- translation;
- embeddings.

This is intentional: the purpose of this metric is to quantify **observable textual variation across generations**.


In [ ]:

# =========================
# TOKENIZATION AND LEVENSHTEIN
# =========================

def tokenize_text(text):
    text = unicodedata.normalize("NFKC", str(text))

    if CONVERT_TO_LOWERCASE:
        text = text.lower()

    if INCLUDE_PUNCTUATION:
        # Unicode words + punctuation as separate tokens.
        tokens = re.findall(
            r"\w+(?:['’]\w+)?|[^\w\s]",
            text,
            flags=re.UNICODE
        )
    else:
        # Words only.
        tokens = re.findall(
            r"\w+(?:['’]\w+)?",
            text,
            flags=re.UNICODE
        )

    return tokens


def normalized_token_levenshtein_distance(tokens_a, tokens_b):
    max_length = max(
        len(tokens_a),
        len(tokens_b)
    )

    if max_length == 0:
        return 0.0

    distance_value = Levenshtein.distance(
        tokens_a,
        tokens_b
    )

    return float(
        distance_value / max_length
    )


# Small implementation sanity checks.
assert normalized_token_levenshtein_distance(
    ["given", "a"],
    ["given", "a"]
) == 0.0

assert np.isclose(
    normalized_token_levenshtein_distance(
        ["given", "a"],
        ["given", "b"]
    ),
    0.5
)

assert normalized_token_levenshtein_distance(
    [],
    ["given"]
) == 1.0

df["tokens"] = df["gherkin"].apply(tokenize_text)
df["n_tokens"] = df["tokens"].apply(len)

print("Tokenization example:")
print(df.iloc[0]["tokens"])
print()
print("Number of tokens:", df.iloc[0]["n_tokens"])


In [ ]:

# =========================
# DISTANCE MATRICES
# NORMALIZED TOKEN-LEVEL LEVENSHTEIN
# =========================

distance_matrices = {}
case_metadata = {}

for case_id, group in df.groupby(
    "case_id",
    sort=False
):

    group = group.sort_values(
        "execution"
    )

    sequences = group["tokens"].tolist()
    N = len(sequences)

    matrix = np.zeros(
        (N, N),
        dtype=float
    )

    for i in range(N):
        for j in range(i + 1, N):

            d = normalized_token_levenshtein_distance(
                sequences[i],
                sequences[j]
            )

            matrix[i, j] = d
            matrix[j, i] = d

    distance_matrices[case_id] = matrix

    case_metadata[case_id] = {
        "original_case":
            group["original_case"].iloc[0]
    }

print(
    "Matrices calculated:",
    len(distance_matrices)
)

print(
    "Dimension of each matrix:",
    next(
        iter(
            distance_matrices.values()
        )
    ).shape
)

print(
    "Observed range:",
    min(m.min() for m in distance_matrices.values()),
    "to",
    max(m.max() for m in distance_matrices.values())
)


In [ ]:

# =========================
# VARIABILITY FUNCTIONS
# =========================

def cumulative_variability_from_matrix(matrix):
    # Returns V_c(n) for n = 2...N.
    #
    # When execution n is added, only
    # the distances between the new execution
    # and the previous n-1 are included.

    N = matrix.shape[0]

    results = {}
    pair_sum = 0.0

    for n in range(2, N + 1):

        new_index = n - 1

        pair_sum += float(
            matrix[
                new_index,
                :new_index
            ].sum()
        )

        pair_count = math.comb(
            n,
            2
        )

        results[n] = (
            pair_sum
            / pair_count
        )

    return results


def bootstrap_mean(
    values,
    n_bootstrap=1000,
    seed=2026,
    confidence=0.95
):
    # Bootstrap over the CASES,
    # not over the pairs within each case.

    values = np.asarray(
        values,
        dtype=float
    )

    rng = np.random.default_rng(
        seed
    )

    samples = rng.choice(
        values,
        size=(
            n_bootstrap,
            len(values)
        ),
        replace=True
    )

    means = samples.mean(
        axis=1
    )

    alpha = 1 - confidence

    return (
        float(
            np.quantile(
                means,
                alpha / 2
            )
        ),
        float(
            np.quantile(
                means,
                1 - alpha / 2
            )
        )
    )


In [ ]:

# =========================
# GLOBAL VARIABILITY CURVE
# =========================

case_variability_by_n = {}

for case_id, matrix in distance_matrices.items():

    case_variability_by_n[
        case_id
    ] = (
        cumulative_variability_from_matrix(
            matrix
        )
    )

curve_rows = []

for n in range(
    2,
    MAX_N + 1
):

    values = np.array(
        [
            case_variability_by_n[c][n]
            for c
            in case_variability_by_n
        ],
        dtype=float
    )

    curve_row = {
        "n_execucoes": n,
        "n_pares_por_caso":
            math.comb(n, 2),

        "media_variabilidade_levenshtein":
            float(values.mean()),

        "mediana_variabilidade_levenshtein":
            float(np.median(values)),

        "desvio_padrao_entre_casos":
            float(values.std(ddof=1)),

        "q1":
            float(
                np.quantile(
                    values,
                    0.25
                )
            ),

        "q3":
            float(
                np.quantile(
                    values,
                    0.75
                )
            ),

        "iqr":
            float(
                np.quantile(
                    values,
                    0.75
                )
                -
                np.quantile(
                    values,
                    0.25
                )
            ),

        "min":
            float(values.min()),

        "max":
            float(values.max())
    }

    if CALCULATE_BOOTSTRAP:

        ci_lower, ci_upper = bootstrap_mean(
            values,
            n_bootstrap=N_BOOTSTRAP,
            seed=SEED_BOOTSTRAP + n
        )

        curve_row[
            "ic95_bootstrap_inferior"
        ] = ci_lower

        curve_row[
            "ic95_bootstrap_superior"
        ] = ci_upper

        curve_row[
            "largura_ic95_bootstrap"
        ] = (
            ci_upper - ci_lower
        )

    curve_rows.append(
        curve_row
    )

curve = pd.DataFrame(
    curve_rows
)

display(curve)


## Stable Convergence Criterion

The reference value is:

\[
V_{final}=V(N)
\]

For each \(n\), the following is calculated:

\[
AbsError(n)=|V(n)-V(N)|
\]

With `ABSOLUTE_TOLERANCE = 0.01`, an estimate is within the range when:

\[
AbsError(n)\le 0.01
\]

However, this is still not sufficient to declare convergence.

The notebook requires that:

1. the candidate is at least `MIN_CONVERGENCE_N`;
2. it is within the tolerance;
3. **all subsequent points through N also remain within the tolerance**;
4. there are at least `MIN_SUBSEQUENT_EXECUTIONS` after the candidate.

The fourth condition prevents the trivial result of declaring the final execution as the convergence point.


In [ ]:

# =========================
# STABLE RETROSPECTIVE CONVERGENCE
# =========================

VARIABILITY_COLUMN = (
    "media_variabilidade_levenshtein"
)

FINAL_VARIABILITY = float(
    curve.loc[
        curve["n_execucoes"] == MAX_N,
        VARIABILITY_COLUMN
    ].iloc[0]
)

curve[
    "erro_absoluto_ao_final"
] = np.abs(
    curve[
        VARIABILITY_COLUMN
    ]
    - FINAL_VARIABILITY
)

curve[
    "dentro_tolerancia"
] = (
    curve[
        "erro_absoluto_ao_final"
    ]
    <= ABSOLUTE_TOLERANCE
)

# Checks, from back to front,
# whether this point and ALL subsequent points
# remain within the tolerance.

flags = (
    curve[
        "dentro_tolerancia"
    ]
    .to_numpy()
)

stable_from_here_on = np.zeros(
    len(flags),
    dtype=bool
)

all_stable = True

for i in range(
    len(flags) - 1,
    -1,
    -1
):

    all_stable = (
        all_stable
        and bool(flags[i])
    )

    stable_from_here_on[i] = (
        all_stable
    )

curve[
    "estavel_daqui_em_diante"
] = stable_from_here_on

curve[
    "execucoes_posteriores_disponiveis"
] = (
    MAX_N
    - curve["n_execucoes"]
)

curve[
    "tem_confirmacao_posterior_suficiente"
] = (
    curve[
        "execucoes_posteriores_disponiveis"
    ]
    >= MIN_SUBSEQUENT_EXECUTIONS
)

candidates = curve[
    (
        curve["n_execucoes"]
        >= MIN_CONVERGENCE_N
    )
    &
    (
        curve[
            "estavel_daqui_em_diante"
        ]
    )
    &
    (
        curve[
            "tem_confirmacao_posterior_suficiente"
        ]
    )
]

if (
    MAX_N
    <
    MIN_CONVERGENCE_N
    + MIN_SUBSEQUENT_EXECUTIONS
):

    CONVERGENCE_POINT = None

    CONVERGENCE_STATUS = (
        f"Not confirmable with N={MAX_N}. "
        f"With MIN_CONVERGENCE_N={MIN_CONVERGENCE_N} "
        f"and MIN_SUBSEQUENT_EXECUTIONS="
        f"{MIN_SUBSEQUENT_EXECUTIONS}, "
        "there is not enough subsequent horizon."
    )

elif len(candidates) == 0:

    CONVERGENCE_POINT = None

    CONVERGENCE_STATUS = (
        f"No confirmed convergence was observed "
        f"through N={MAX_N} with absolute tolerance "
        f"of {ABSOLUTE_TOLERANCE:.4f} and "
        f"{MIN_SUBSEQUENT_EXECUTIONS} "
        "minimum subsequent executions."
    )

else:

    CONVERGENCE_POINT = int(
        candidates.iloc[0][
            "n_execucoes"
        ]
    )

    CONVERGENCE_STATUS = (
        f"Stable convergence confirmed "
        f"at N={CONVERGENCE_POINT}, "
        f"using V({MAX_N}) as the reference, "
        f"absolute tolerance "
        f"{ABSOLUTE_TOLERANCE:.4f} and "
        f"at least "
        f"{MIN_SUBSEQUENT_EXECUTIONS} "
        "subsequent executions."
    )

print(
    "Final variability "
    "(Normalized Token-Level Levenshtein):",
    round(FINAL_VARIABILITY, 6)
)

print(
    CONVERGENCE_STATUS
)

display(
    curve[
        [
            "n_execucoes",
            VARIABILITY_COLUMN,
            "erro_absoluto_ao_final",
            "dentro_tolerancia",
            "estavel_daqui_em_diante",
            "execucoes_posteriores_disponiveis",
            "tem_confirmacao_posterior_suficiente"
        ]
    ]
)


In [ ]:

# =========================
# SENSITIVITY ANALYSIS
# OF THE TOLERANCE
# =========================

def find_convergence_for_tolerance(
    base_curve,
    tolerance
):
    temp = base_curve.copy()

    temp["ok"] = (
        temp["erro_absoluto_ao_final"]
        <= tolerance
    )

    flags = temp["ok"].to_numpy()

    stable = np.zeros(
        len(flags),
        dtype=bool
    )

    all_stable = True

    for i in range(
        len(flags) - 1,
        -1,
        -1
    ):
        all_stable = (
            all_stable
            and bool(flags[i])
        )
        stable[i] = all_stable

    temp["estavel"] = stable

    local_candidates = temp[
        (
            temp["n_execucoes"]
            >= MIN_CONVERGENCE_N
        )
        &
        temp["estavel"]
        &
        (
            temp[
                "execucoes_posteriores_disponiveis"
            ]
            >= MIN_SUBSEQUENT_EXECUTIONS
        )
    ]

    if len(local_candidates) == 0:
        return None

    return int(
        local_candidates.iloc[0][
            "n_execucoes"
        ]
    )


sensitivity = pd.DataFrame(
    [
        {
            "tolerancia_absoluta":
                tolerance_value,

            "ponto_convergencia":
                find_convergence_for_tolerance(
                    curve,
                    tolerance_value
                )
        }
        for tolerance_value
        in SENSITIVITY_TOLERANCES
    ]
)

display(sensitivity)


In [ ]:

# =========================
# FINAL SUMMARY PER CASE
# =========================

case_rows = []

for case_id, matrix in distance_matrices.items():

    upper_triangle = matrix[
        np.triu_indices(
            MAX_N,
            k=1
        )
    ]

    case_rows.append({
        "case_id":
            case_id,

        "original_case":
            case_metadata[
                case_id
            ][
                "original_case"
            ],

        "n_execucoes":
            MAX_N,

        "n_pares":
            len(upper_triangle),

        "levenshtein_tokens_norm_media":
            float(
                np.mean(
                    upper_triangle
                )
            ),

        "levenshtein_tokens_norm_mediana":
            float(
                np.median(
                    upper_triangle
                )
            ),

        "levenshtein_tokens_norm_desvio_padrao":
            (
                float(
                    np.std(
                        upper_triangle,
                        ddof=1
                    )
                )
                if len(upper_triangle) > 1
                else 0.0
            ),

        "levenshtein_tokens_norm_min":
            float(
                np.min(
                    upper_triangle
                )
            ),

        "levenshtein_tokens_norm_max":
            float(
                np.max(
                    upper_triangle
                )
            ),

        "levenshtein_tokens_norm_iqr":
            float(
                np.quantile(
                    upper_triangle,
                    0.75
                )
                -
                np.quantile(
                    upper_triangle,
                    0.25
                )
            )
    })

case_variability = (
    pd.DataFrame(
        case_rows
    )
    .sort_values(
        "levenshtein_tokens_norm_media",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

display(
    case_variability.head(
        TOP_CASES
    )
)


In [ ]:

# =========================
# GLOBAL SUMMARY
# =========================

final_values = (
    case_variability[
        "levenshtein_tokens_norm_media"
    ]
    .to_numpy()
)

global_summary = {
    "model":
        llm_model,

    "technique":
        technique,

    "metric":
        "normalized_token_level_levenshtein_distance",

    "metric_range":
        [0.0, 1.0],

    "tokenization": {
        "unicode_normalization":
            "NFKC",

        "lowercase":
            CONVERT_TO_LOWERCASE,

        "punctuation_as_tokens":
            INCLUDE_PUNCTUATION
    },

    "number_of_cases":
        int(
            len(
                case_variability
            )
        ),

    "number_of_executions":
        int(
            MAX_N
        ),

    "pairs_per_case":
        int(
            math.comb(
                MAX_N,
                2
            )
        ),

    "global_mean_case_variability":
        float(
            final_values.mean()
        ),

    "global_median_case_variability":
        float(
            np.median(
                final_values
            )
        ),

    "global_sd_between_cases":
        float(
            final_values.std(
                ddof=1
            )
        ),

    "global_iqr_between_cases":
        float(
            np.quantile(
                final_values,
                0.75
            )
            -
            np.quantile(
                final_values,
                0.25
            )
        ),

    "convergence": {
        "method":
            "retrospective_stable_absolute_difference_to_final_estimate",

        "minimum_n":
            int(
                MIN_CONVERGENCE_N
            ),

        "absolute_tolerance":
            float(
                ABSOLUTE_TOLERANCE
            ),

        "minimum_posterior_executions":
            int(
                MIN_SUBSEQUENT_EXECUTIONS
            ),

        "reference_n":
            int(
                MAX_N
            ),

        "reference_variability":
            float(
                FINAL_VARIABILITY
            ),

        "convergence_point":
            CONVERGENCE_POINT,

        "status":
            CONVERGENCE_STATUS
    }
}

if CALCULATE_BOOTSTRAP:

    ci_lower, ci_upper = bootstrap_mean(
        final_values,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED_BOOTSTRAP
    )

    global_summary[
        "global_mean_bootstrap_ci95"
    ] = {
        "lower":
            ci_lower,

        "upper":
            ci_upper
    }

print(
    json.dumps(
        global_summary,
        ensure_ascii=False,
        indent=2
    )
)


In [ ]:

# =========================
# CONVERGENCE PLOT
# =========================

plt.figure(
    figsize=(10, 6)
)

plt.plot(
    curve[
        "n_execucoes"
    ],
    curve[
        "media_variabilidade_levenshtein"
    ],
    marker="o",
    label="Cumulative variability"
)

plt.axhline(
    FINAL_VARIABILITY,
    linestyle="--",
    label=f"Final estimate (N={MAX_N})"
)

lower_limit = max(
    0.0,
    FINAL_VARIABILITY
    - ABSOLUTE_TOLERANCE
)

upper_limit = min(
    1.0,
    FINAL_VARIABILITY
    + ABSOLUTE_TOLERANCE
)

plt.axhline(
    lower_limit,
    linestyle=":",
    label=(
        f"Limit -"
        f"{ABSOLUTE_TOLERANCE:.3f}"
    )
)

plt.axhline(
    upper_limit,
    linestyle=":",
    label=(
        f"Limit +"
        f"{ABSOLUTE_TOLERANCE:.3f}"
    )
)

if CONVERGENCE_POINT is not None:

    plt.axvline(
        CONVERGENCE_POINT,
        linestyle="--",
        label=(
            f"Convergence: "
            f"N={CONVERGENCE_POINT}"
        )
    )

plt.xlabel(
    "Cumulative number of executions"
)

plt.ylabel(
    "Normalized Token-Level "
    "Mean Levenshtein Distance"
)

plt.title(
    f"Textual variability convergence\n"
    f"{llm_model} | {technique}"
)

plt.ylim(
    0,
    min(
        1.0,
        max(
            curve[
                "media_variabilidade_levenshtein"
            ].max()
            + 0.05,
            upper_limit
            + 0.02
        )
    )
)

plt.grid(
    alpha=0.3
)

plt.legend()

plt.show()


In [ ]:

# =========================
# FINAL VARIABILITY DISTRIBUTION
# =========================

plt.figure(
    figsize=(10, 6)
)

plt.hist(
    case_variability[
        "levenshtein_tokens_norm_media"
    ],
    bins=25
)

plt.xlabel(
    "Normalized Token-Level "
    "Mean Levenshtein Distance"
)

plt.ylabel(
    "Number of cases"
)

plt.title(
    f"Textual variability per case\n"
    f"{llm_model} | {technique} | N={MAX_N}"
)

plt.xlim(
    0,
    1
)

plt.grid(
    alpha=0.3
)

plt.show()


In [ ]:

# =========================
# MOST VARIABLE CASES
# =========================

top_cases = (
    case_variability
    .head(
        TOP_CASES
    )
    .sort_values(
        "levenshtein_tokens_norm_media",
        ascending=True
    )
)

plt.figure(
    figsize=(
        10,
        max(
            6,
            TOP_CASES * 0.35
        )
    )
)

plt.barh(
    top_cases[
        "case_id"
    ],
    top_cases[
        "levenshtein_tokens_norm_media"
    ]
)

plt.xlabel(
    "Normalized Token-Level "
    "Mean Levenshtein Distance"
)

plt.ylabel(
    "Case"
)

plt.title(
    f"{TOP_CASES} cases "
    "with the highest textual variability"
)

plt.xlim(
    0,
    1
)

plt.grid(
    axis="x",
    alpha=0.3
)

plt.show()


In [ ]:

# =========================
# SAVING
# =========================

def slug(text):
    text = str(
        text
    ).lower()

    text = re.sub(
        r"[^a-z0-9]+",
        "-",
        text
    )

    return (
        text
        .strip("-")[:80]
    )


output_base = (
    f"{slug(llm_model)}_"
    f"{slug(technique)}"
)

curve_file = (
    "convergencia_levenshtein_tokens_"
    f"{output_base}.csv"
)

cases_file = (
    "variabilidade_levenshtein_tokens_por_caso_"
    f"{output_base}.csv"
)

json_file = (
    "resumo_variabilidade_levenshtein_tokens_"
    f"{output_base}.json"
)

curve.to_csv(
    curve_file,
    index=False,
    encoding="utf-8-sig"
)

case_variability.to_csv(
    cases_file,
    index=False,
    encoding="utf-8-sig"
)

json_output = {
    "summary":
        global_summary,

    "sensitivity_analysis":
        sensitivity.to_dict(
            orient="records"
        ),

    "convergence_curve":
        curve.to_dict(
            orient="records"
        ),

    "case_variability":
        case_variability.to_dict(
            orient="records"
        )
}

with open(
    json_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        json_output,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    "Generated files:"
)

print(
    "-",
    curve_file
)

print(
    "-",
    cases_file
)

print(
    "-",
    json_file
)

# In Colab, if you want to download:
#
# from google.colab import files
# files.download(arquivo_curva)
# files.download(arquivo_casos)
# files.download(arquivo_json)


# Interpretation

## 1. Textual Variability

The main indicator per case is:

`levenshtein_tokens_norm_media`

This output column name is intentionally preserved from the original notebook for compatibility with previously generated files.

It is on the \([0,1]\) scale:

- closer to `0` → more textually stable generations;
- higher → more textually different generations.

Example:

`0.18`

means an **average normalized edit distance of 0.18** among the generations of that case.

Avoid writing simply “18% semantically different.” The metric does not measure semantics.

## 2. What Counts as a Difference

By default:

- `Given` vs `When` counts as a substitution;
- an additional word counts as an insertion;
- a missing word counts as a deletion;
- different synonyms count as different tokens;
- differences only in uppercase/lowercase are ignored;
- spaces/indentation do not count;
- punctuation counts because `INCLUDE_PUNCTUATION = True`.

## 3. Comparison of the 5 Models × 3 Techniques

Run the same notebook for all 15 JSON files.

The main global indicator is:

`global_mean_case_variability`

Because all configurations use the same metric on the \([0,1]\) scale, the values can be compared directly across the experimental configurations.

Also report:

- `global_median_case_variability`;
- `global_sd_between_cases`;
- `global_iqr_between_cases`;
- 95% bootstrap CI of the global mean.

## 4. Convergence

`convergence_point` is the smallest number of executions from which:

1. the estimate remained within the absolute tolerance relative to \(V(N)\);
2. it never left that range again through the final execution;
3. there were enough subsequent executions to confirm stability.

Example:

- `reference_n = 20`;
- `convergence_point = 11`;
- `absolute_tolerance = 0.01`;
- `minimum_posterior_executions = 5`.

Interpretation:

> From 11 executions onward, the cumulative estimate of textual variability remained at most 0.01 units away from the estimate obtained with 20 executions, maintaining this behavior throughout the subsequent observed executions.

## 5. A Single Number of Repetitions for the Study

After running all 15 files:

\[
N_{final}
=
\max(
N_{\text{convergence of the 15 configurations}}
)
\]

If any configuration does not show confirmed convergence within the available horizon, increase the number of executions for that analysis before defining \(N_{final}\).

## 6. Tolerance Sensitivity

The `sensitivity` Python variable displays the convergence point for different tolerances, by default:

- 0.005;
- 0.010;
- 0.020.

The exported column names remain identical to those of the original notebook for compatibility with previously generated output files.
